# Analytics Made Simple — Interactive SQL Playground
This notebook lets you run the entire 14-part SQL curriculum interactively in Python using an embedded **SQLite** engine. No external database server required!


In [ ]:
import sqlite3
import pandas as pd

# Connect to in-memory playground database
conn = sqlite3.connect(":memory:")

# Load the full playground schema and sample data
with open("01_playground_setup.sql", "r") as f:
    schema_script = f.read()

conn.executescript(schema_script)
print("✅ Database initialized successfully with customers, orders, and order_lines!")


### Query 1: Customer Overview and Order History


In [ ]:
query = '''
SELECT 
    c.name AS customer_name,
    c.region,
    o.order_id,
    o.order_date,
    o.status,
    o.order_total
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
ORDER BY o.order_id;
'''
pd.read_sql_query(query, conn)


### Query 2: Regional Revenue Performance (CTE)


In [ ]:
query_cte = '''
WITH regional_performance AS (
    SELECT 
        c.region,
        COUNT(o.order_id) AS order_volume,
        SUM(o.order_total) AS regional_revenue
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE o.status = 'completed'
    GROUP BY c.region
)
SELECT 
    region,
    order_volume,
    regional_revenue,
    ROUND(regional_revenue * 100.0 / (SELECT SUM(regional_revenue) FROM regional_performance), 2) AS pct_of_total
FROM regional_performance
ORDER BY regional_revenue DESC;
'''
pd.read_sql_query(query_cte, conn)


### Query 3: Window Functions (Rank within Region)


In [ ]:
query_window = '''
SELECT 
    o.order_id,
    c.region,
    c.name,
    o.order_total,
    ROW_NUMBER() OVER (PARTITION BY c.region ORDER BY o.order_total DESC) AS rank_in_region
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.status = 'completed';
'''
pd.read_sql_query(query_window, conn)
